In [ ]:
# make local editable packages automatically reload
%load_ext autoreload
%autoreload 2
    
# Import dependencies
import numpy as np
import omnirefactor

# set up plotting defaults
from omnirefactor.plot import imshow
omnirefactor.plot.setup()
%matplotlib inline

# import hiprpy
import ncolor
import edt

In [ ]:
import ocdkit.spatial
masks = ocdkit.spatial.nd_grid_hypercube_labels((1000,1000), 10)

In [ ]:
ncolor.label(masks)

In [ ]:
# %timeit ncolor.label(masks)

In [ ]:
import edt
from edt import expand_labels
out = expand_labels(masks+1)
np.unique(out)

In [ ]:
def random_hypervoxels(side, dim, M=None, seed=None, dtype=np.int32):
    """
    ND label array with M single-voxel labels.
    Shape = (side,)*dim. Labels are 1..M. Background is 0.
    """
    if side <= 0 or dim <= 0:
        raise ValueError("side and dim must be positive")
    total = side ** dim
    if M is None:
        M = total // 2
    if M < 0 or M > total:
        raise ValueError("M must be in [0, side**dim]")


    rng = np.random.default_rng(seed)
    flat = rng.choice(total, size=M, replace=False)
    coords = np.unravel_index(flat, (side,) * dim)

    arr = np.zeros((side,) * dim, dtype=dtype)
    arr[coords] = np.arange(1, M + 1, dtype=dtype)
    return arr

labels = random_hypervoxels(2000,2)

# %timeit ncolor.expand_labels(labels)
%timeit expand_labels(labels, parallel = -1)

In [ ]:
# %timeit edt.edt(labels)
%timeit edt.edt(labels, parallel=-1)
np.sum(expand_labels(labels, parallel = -1) != ncolor.expand_labels(labels))/labels.size


In [ ]:
# from omnirefactor.io import imread, adjust_file_path
# f = '/Volumes/DataDrive/omnipose_link_train/combined/wiggins/wiggins_ensemble_18_masks.tif'
# # f = '/Volumes/HiprDrive/HiPR-Map/UPLP/20250806_UPLPPanelChck_s2/s2/2025_08_06_UPLPPanelChck_fov00_s2_p1X_fov8_seg.npy'
# f = adjust_file_path(f)
# labels = imread(f)
# e1 = ncolor.expand_labels(labels)
# e2 = expand_labels(labels)

# ties = ['last', 'first', 'scipy', 'left', 'smallest']
# e2 = expand_labels(labels)
# # e2 = expand_labels(labels, anisotropy=None, preserve_existing=True, ties='left')

# # e3 = expand_labels(labels,  ties=ties[1])



# def shuffle_labels(mask, keep_background=True, seed=42):
#     """
#     Randomly permute instance labels in a segmentation mask.
#     mask: integer array, 0 reserved for background
#     keep_background: if True, label 0 stays 0
#     seed: optional RNG seed
#     """
#     rng = np.random.default_rng(seed)
#     labels = np.unique(mask)
#     if keep_background and 0 in labels:
#         labels = labels[labels != 0]
#         shuffled = rng.permutation(labels)
#         mapping = {old: new for old, new in zip(labels, shuffled)}
#         mapping[0] = 0
#     else:
#         shuffled = rng.permutation(labels)
#         mapping = {old: new for old, new in zip(labels, shuffled)}
#     # apply mapping
#     out = np.zeros_like(mask)
#     for old, new in mapping.items():
#         out[mask == old] = new
#     return out

# sl = shuffle_labels(labels)
# e3 = expand_labels(labels, ties=ties[1])

# e4 = ncolor.expand_labels(sl)

# m1 = e1*(labels>0)
# m2 = e2*(labels>0)

# print(np.all(e1==e2), np.sum(e1!=e2), np.any(e2))
# print(np.all(m1==m2), np.sum(m1!=m2))
# print('a',np.all(e1==e3), np.sum(e1!=e3))
# print(np.sum(expand_labels(labels,  ties=ties[0])!=expand_labels(labels,  ties=ties[3])))
# print(np.sum(expand_labels(labels,  ties=ties[0])!=expand_labels(labels,  ties=ties[2])))

# ncl = [omnirefactor.plot.apply_ncolor(i) for i in [labels, sl, e1, e2, e3]]
# imshow(ncl+[e1!=e2]+[e1!=e3]+[(ncl[-1]!=ncl[-3]).sum(axis=-1)],4)


from omnirefactor.io import imread, adjust_file_path
f = '/Volumes/DataDrive/omnipose_link_train/combined/wiggins/wiggins_ensemble_18_masks.tif'
# f = '/Volumes/HiprDrive/HiPR-Map/UPLP/20250806_UPLPPanelChck_s2/s2/2025_08_06_UPLPPanelChck_fov00_s2_p1X_fov8_seg.npy'
f = adjust_file_path(f)
labels = imread(f)
e1 = ncolor.expand_labels(labels)
e2 = expand_labels(labels)

ties = ['last', 'first', 'scipy', 'left', 'smallest']
e2 = expand_labels(labels)

m1 = e1*(labels>0)
m2 = e2*(labels>0)

print(np.all(e1==e2), np.sum(e1!=e2), np.any(e2))
print(np.all(m1==m2), np.sum(m1!=m2))

ncl = [omnirefactor.plot.apply_ncolor(i) for i in [labels, e1, e2]]
imshow(ncl,4)

In [ ]:
# imshow(shuffle_labels(labels))
# (ncl[-1]!=ncl[-3]).sum(axis=-1).shape
e2.any()

In [ ]:
%timeit ncolor.expand_labels(labels)
%timeit expand_labels(labels, parallel = -1)

In [ ]:
# %load_ext line_profiler
# %lprun -f ncolor.label ncolor.label(masks)
# %lprun -f ncolor.get_lut ncolor.label(masks)

# # %lprun -f ncolor.render_net ncolor.label(masks)
# # %lprun -f ncolor.mapidx ncolor.label(masks)
# %lprun -f ncolor.connect ncolor.label(masks)

imshow(edt.edt(labels))

In [ ]:
from scipy.ndimage import distance_transform_edt
dt, nearest_label_coords = distance_transform_edt(labels==0, 
                                              return_distances=True, 
                                              return_indices=True)
expanded_labels, dt2 = edt.feature_transform(labels==0, 
                                             return_distances=1, 
                                             # ties='scipy'
                                            )
# dt2 = edt.edt(labels==0)
print(dt2.any())
# imshow(dt)

imshow([dt, dt2**0.5,np.abs(dt-dt2**0.5)])
dt.max(), dt2.max(), dt.min(), dt2.min()
edt nbroken by a square root

In [ ]:
np.abs(dt-dt2**0.5).max()